# Compare CNN-VAE Jet Image Reconstructions

Run this notebook from the `vae_cnn/` directory. It loads one trained CNN-VAE checkpoint, builds event-level pT-normalized jet images from raw `.dat` events, and compares input images with reconstructed images.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd()
CODE_DIR = PROJECT_ROOT / "code_training"
sys.path.insert(0, str(CODE_DIR))

from betaVae import BetaVAE
from jet_image_utils import load_events_from_dat, event_to_jet_image, event_to_jet_image_no_pid, pt_normalize, build_pid_channel_map, build_pid_lookup

plt.rcParams["figure.dpi"] = 120
torch.set_grad_enabled(False)
PROJECT_ROOT

## Configuration

Change `CKPT_PATH` to inspect another trained model. The data path defaults to `ai4jet/data/u_jet_CMS_sample.dat` when this notebook is run from `vae_cnn/`.

In [ ]:
CKPT_PATH = PROJECT_ROOT / "results" / "200GeVjet_nopid_annealing/mse_all" / "beta_0p000001" / "best_model_betavae.pth"
DATA_PATH = PROJECT_ROOT.parent / "data" / "u_jet_real_sample.dat"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "data" / "u_jet_real_sample.dat"

EVENT_INDEX = 0
DETERMINISTIC = True  # True: decode(mu); False: sample z from q(z|x)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

assert CKPT_PATH.exists(), f"Checkpoint not found: {CKPT_PATH}"
assert DATA_PATH.exists(), f"Data file not found: {DATA_PATH}"
print("checkpoint:", CKPT_PATH)
print("data      :", DATA_PATH)
print("device    :", DEVICE)

In [ ]:
def pid_map_from_checkpoint_or_events(checkpoint, events):
    if "pid_channel_map" in checkpoint:
        return {int(pid): int(ch) for pid, ch in checkpoint["pid_channel_map"].items()}
    return build_pid_channel_map(events)


def checkpoint_hidden_dims(checkpoint):
    return list(checkpoint.get("hidden_dims", [32, 64, 128, 256, 512]))


def checkpoint_output_mode(checkpoint):
    if "output_mode" in checkpoint:
        return str(checkpoint["output_mode"])
    if checkpoint.get("normalization", None) == "event_pt":
        return "event_pt"
    return "tanh"


def checkpoint_scalar(checkpoint, key, default):
    value = checkpoint.get(key, default)
    if hasattr(value, "item"):
        value = value.item()
    return float(value)


def checkpoint_bool(checkpoint, key, default):
    value = checkpoint.get(key, default)
    if hasattr(value, "item"):
        value = value.item()
    return bool(value)


def checkpoint_eta_max(checkpoint):
    return checkpoint_scalar(checkpoint, "eta_max", 0.5)


def checkpoint_phi_max(checkpoint):
    return checkpoint_scalar(checkpoint, "phi_max", 0.5)


def checkpoint_recenter_output(checkpoint):
    return checkpoint_bool(checkpoint, "recenter_output", True)


def image_extent(checkpoint):
    return [-checkpoint_eta_max(checkpoint), checkpoint_eta_max(checkpoint),
            -checkpoint_phi_max(checkpoint), checkpoint_phi_max(checkpoint)]


def checkpoint_channel_mode(checkpoint):
    if "channel_mode" in checkpoint:
        return str(checkpoint["channel_mode"])
    pid_map = checkpoint.get("pid_channel_map", {})
    pid_map = {int(pid): int(ch) for pid, ch in pid_map.items()} if hasattr(pid_map, "items") else {}
    if int(checkpoint["n_channels"]) == 1 and (not pid_map or pid_map == {0: 0}):
        return "eta_phi_total_pt_no_pid"
    return "pid_channels"


def checkpoint_uses_no_pid_images(checkpoint):
    return checkpoint_channel_mode(checkpoint) == "eta_phi_total_pt_no_pid"


def checkpoint_uses_standardized_images(checkpoint):
    return "mean_img" in checkpoint and "std_img" in checkpoint and checkpoint.get("normalization", None) != "event_pt"


def load_model(checkpoint, device):
    model = BetaVAE(
        in_channels=int(checkpoint["n_channels"]),
        latent_dim=int(checkpoint["latent_dim"]),
        image_size=int(checkpoint["img_size"]),
        hidden_dims=checkpoint_hidden_dims(checkpoint),
        output_mode=checkpoint_output_mode(checkpoint),
        recenter_output=checkpoint_recenter_output(checkpoint),
        eta_max=checkpoint_eta_max(checkpoint),
        phi_max=checkpoint_phi_max(checkpoint),
    ).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model


def make_event_image(event, pid_lookup, pid_min, n_channels, img_size, checkpoint):
    eta_max = checkpoint_eta_max(checkpoint)
    phi_max = checkpoint_phi_max(checkpoint)
    if checkpoint_uses_no_pid_images(checkpoint):
        img = event_to_jet_image_no_pid(
            event,
            eta_bins=img_size, phi_bins=img_size,
            eta_max=eta_max, phi_max=phi_max,
        )
    else:
        img = event_to_jet_image(
            event, pid_lookup, pid_min, n_channels,
            eta_bins=img_size, phi_bins=img_size,
            eta_max=eta_max, phi_max=phi_max,
        )
    return pt_normalize(img).astype(np.float32)


def model_input_from_event_image(img, checkpoint):
    if checkpoint_uses_standardized_images(checkpoint):
        mean_img = np.asarray(checkpoint["mean_img"], dtype=np.float32)
        std_img = np.asarray(checkpoint["std_img"], dtype=np.float32)
        return ((img - mean_img) / (std_img + 1e-5)).astype(np.float32)
    return img.astype(np.float32)


def event_image_from_model_output(output, checkpoint):
    if checkpoint_uses_standardized_images(checkpoint):
        mean_img = np.asarray(checkpoint["mean_img"], dtype=np.float32)
        std_img = np.asarray(checkpoint["std_img"], dtype=np.float32)
        output = output * (std_img + 1e-5) + mean_img
        output = np.clip(output, 0.0, None)
        output = pt_normalize(output)
    return output.astype(np.float32)


def reconstruct_image(model, img, checkpoint, device, deterministic=True):
    model_input = model_input_from_event_image(img, checkpoint)
    x = torch.from_numpy(model_input[None]).to(device)
    with torch.no_grad():
        mu, logvar = model.encode(x)
        z = mu if deterministic else model.reparameterize(mu, logvar)
        recon_raw = model.decode(z).cpu().numpy()[0]
    recon = event_image_from_model_output(recon_raw, checkpoint)
    return recon, mu.cpu().numpy()[0], logvar.cpu().numpy()[0]


def image_metrics(img, recon):
    diff = recon - img
    return {
        "sse": float(np.sum(diff ** 2)),
        "mse_per_pixel": float(np.mean(diff ** 2)),
        "mae_per_pixel": float(np.mean(np.abs(diff))),
        "input_sum": float(img.sum()),
        "recon_sum": float(recon.sum()),
        "input_max": float(img.max()),
        "recon_max": float(recon.max()),
    }


In [ ]:
def zero_as_white_cmap(name="viridis"):
    cmap = plt.get_cmap(name).copy()
    cmap.set_bad("white")
    return cmap


PT_CMAP = zero_as_white_cmap()
WHITE_FLOOR = 1e-3  # pT-fraction values <= this are drawn as white; increase if output backgrounds remain dark


def mask_zero_pixels(array, floor=WHITE_FLOOR):
    return np.ma.masked_where(array <= floor, array)


def plot_total_images(img, recon, title_prefix=""):
    total_in = img.sum(axis=0)
    total_re = recon.sum(axis=0)
    diff = total_re - total_in
    vmax = max(float(total_in.max()), float(total_re.max()), 1e-12)
    dmax = max(abs(float(diff.min())), abs(float(diff.max())), 1e-12)

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), constrained_layout=True)
    im0 = axes[0].imshow(mask_zero_pixels(total_in), origin="lower", extent=image_extent(checkpoint), cmap=PT_CMAP, vmin=0, vmax=vmax)
    axes[0].set_title("Input, sum over channels")
    axes[1].imshow(mask_zero_pixels(total_re), origin="lower", extent=image_extent(checkpoint), cmap=PT_CMAP, vmin=0, vmax=vmax)
    axes[1].set_title("Reconstruction, sum over channels")
    im2 = axes[2].imshow(diff, origin="lower", extent=image_extent(checkpoint), cmap="coolwarm", vmin=-dmax, vmax=dmax)
    axes[2].set_title("Recon - input")
    for ax in axes:
        ax.set_xlabel(r"$\Delta\eta$")
        ax.set_ylabel(r"$\Delta\phi$")
    fig.colorbar(im0, ax=axes[:2], shrink=0.85, label="pT fraction")
    fig.colorbar(im2, ax=axes[2], shrink=0.85, label="difference")
    if title_prefix:
        fig.suptitle(title_prefix)
    return fig, axes


def plot_dominant_channels(img, recon, pid_channel_map):
    channel_label = "total pT channel" if checkpoint_uses_no_pid_images(checkpoint) else None
    pid_by_channel = {ch: pid for pid, ch in pid_channel_map.items()}
    input_ch = int(img.sum(axis=(1, 2)).argmax())
    recon_ch = int(recon.sum(axis=(1, 2)).argmax())
    channels = [input_ch, recon_ch] if input_ch != recon_ch else [input_ch]

    fig, axes = plt.subplots(len(channels), 2, figsize=(8, 3.2 * len(channels)), squeeze=False, constrained_layout=True)
    for row, ch in enumerate(channels):
        vmax = max(float(img[ch].max()), float(recon[ch].max()), 1e-12)
        axes[row, 0].imshow(mask_zero_pixels(img[ch]), origin="lower", extent=image_extent(checkpoint), cmap=PT_CMAP, vmin=0, vmax=vmax)
        axes[row, 1].imshow(mask_zero_pixels(recon[ch]), origin="lower", extent=image_extent(checkpoint), cmap=PT_CMAP, vmin=0, vmax=vmax)
        label = channel_label or f"PID={pid_by_channel.get(ch, '?')}"
        axes[row, 0].set_title(f"Input channel {ch}, {label}")
        axes[row, 1].set_title(f"Recon channel {ch}, {label}")
    return fig, axes


## Load checkpoint, model, and events

In [ ]:
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
events = load_events_from_dat(str(DATA_PATH))
pid_channel_map = pid_map_from_checkpoint_or_events(checkpoint, events)
pid_lookup, pid_min = build_pid_lookup(pid_channel_map)
model = load_model(checkpoint, DEVICE)

img_size = int(checkpoint["img_size"])
n_channels = int(checkpoint["n_channels"])

print("events      :", len(events))
print("img_size    :", img_size)
print("n_channels  :", n_channels)
print("latent_dim  :", int(checkpoint["latent_dim"]))
print("hidden_dims :", checkpoint_hidden_dims(checkpoint))
print("output_mode :", checkpoint_output_mode(checkpoint))
print("recenter   :", checkpoint_recenter_output(checkpoint))
print("eta_max    :", checkpoint_eta_max(checkpoint))
print("phi_max    :", checkpoint_phi_max(checkpoint))
print("channel_mode:", checkpoint_channel_mode(checkpoint))
print("legacy std  :", checkpoint_uses_standardized_images(checkpoint))
print("val_loss    :", checkpoint.get("val_loss"))


## Single event reconstruction

In [ ]:
event = events[EVENT_INDEX]
event = events[2]
img = make_event_image(event, pid_lookup, pid_min, n_channels, img_size, checkpoint)
recon, mu, logvar = reconstruct_image(model, img, checkpoint, DEVICE, deterministic=DETERMINISTIC)

print("event index:", EVENT_INDEX)
print("n_particles:", len(event))
for key, value in image_metrics(img, recon).items():
    print(f"{key:>14s}: {value:.6g}")

plot_total_images(img, recon, title_prefix=f"{CKPT_PATH.parent.name}, event {EVENT_INDEX}")
plt.show()
plot_dominant_channels(img, recon, pid_channel_map)
plt.show()

## Scan several events

In [ ]:
START = 0
N_SHOW = 5
DIFF_VMAX = 0.05  # max absolute value in recon-input difference images; adjust for better color contrast if needed

rows = []
fig, axes = plt.subplots(N_SHOW, 3, figsize=(11, 2.8 * N_SHOW), constrained_layout=True)
for r in range(N_SHOW):
    idx = START + r
    img_r = make_event_image(events[idx], pid_lookup, pid_min, n_channels, img_size, checkpoint)
    recon_r, _, _ = reconstruct_image(model, img_r, checkpoint, DEVICE, deterministic=True)
    rows.append({"event": idx, **image_metrics(img_r, recon_r)})

    total_in = img_r.sum(axis=0)
    total_re = recon_r.sum(axis=0)
    diff = total_re - total_in
    vmax = max(float(total_in.max()), float(total_re.max()), 1e-12)
    dmax = max(abs(float(diff.min())), abs(float(diff.max())), 1e-12)
    axes[r, 0].imshow(mask_zero_pixels(total_in), origin="lower", extent=image_extent(checkpoint), cmap=PT_CMAP, vmin=0, vmax=vmax)
    axes[r, 1].imshow(mask_zero_pixels(total_re), origin="lower", extent=image_extent(checkpoint), cmap=PT_CMAP, vmin=0, vmax=vmax)
    #axes[r, 2].imshow(diff, origin="lower", extent=image_extent(checkpoint), cmap="coolwarm", vmin=-dmax, vmax=dmax)
    axes[r, 2].imshow(diff, cmap="coolwarm", vmin=-DIFF_VMAX, vmax=DIFF_VMAX)
    axes[r, 0].set_ylabel(f"event {idx}")

axes[0, 0].set_title("Input")
axes[0, 1].set_title("Reconstruction")
axes[0, 2].set_title("Recon - input")
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

rows


In [ ]:
# Active-unit analysis across a reproducible event sample.
# A latent feature is active when Var_x[mu_i(x)] is above the chosen threshold.
LATENT_N_EVENTS = min(100000, len(events))
LATENT_BATCH_SIZE = 256
LATENT_RANDOM_SEED = 0
ACTIVE_THRESHOLD = 1e-2

rng = np.random.default_rng(LATENT_RANDOM_SEED)
latent_event_indices = rng.choice(len(events), size=LATENT_N_EVENTS, replace=False)
latent_mu_batches = []
latent_logvar_batches = []

for start in range(0, LATENT_N_EVENTS, LATENT_BATCH_SIZE):
    batch_indices = latent_event_indices[start:start + LATENT_BATCH_SIZE]
    batch_images = np.stack([
        model_input_from_event_image(
            make_event_image(events[i], pid_lookup, pid_min, n_channels, img_size, checkpoint),
            checkpoint,
        )
        for i in batch_indices
    ])
    x_batch = torch.from_numpy(batch_images).to(DEVICE)
    with torch.no_grad():
        mu_batch, logvar_batch = model.encode(x_batch)
    latent_mu_batches.append(mu_batch.cpu().numpy())
    latent_logvar_batches.append(logvar_batch.cpu().numpy())

latent_mu = np.concatenate(latent_mu_batches, axis=0)
latent_logvar = np.concatenate(latent_logvar_batches, axis=0)
activity_scores = latent_mu.var(axis=0)
latent_features = np.arange(activity_scores.size)
is_active = activity_scores > ACTIVE_THRESHOLD
bar_colors = np.where(is_active, "tab:blue", "tab:gray")

fig, ax = plt.subplots(figsize=(13, 4.8), constrained_layout=True)
ax.bar(latent_features, activity_scores, color=bar_colors, edgecolor="black", linewidth=0.4)
ax.axhline(
    ACTIVE_THRESHOLD, color="tab:red", linestyle="--", linewidth=1.2,
    label=rf"active threshold = {ACTIVE_THRESHOLD:g}",
)
ax.set_yscale("log")
ax.set_xticks(latent_features)
ax.set_xlabel("latent feature index")
ax.set_ylabel(r"activity score $\mathrm{Var}_x[\mu_i(x)]$ (log scale)")
ax.set_title(
    f"Latent active units: {is_active.sum()}/{activity_scores.size} active "
    f"({LATENT_N_EVENTS:,} events)"
)
ax.grid(axis="y", which="both", alpha=0.25)
ax.legend()

print("active features  :", latent_features[is_active].tolist())
print("inactive features:", latent_features[~is_active].tolist())
plt.show()

In [ ]:
# Latent signal-to-noise ratio: between-jet variation divided by posterior sampling noise.
# sigma_i^2(x) = exp(logvar_i(x)); this ratio is dimensionless.
# SNR_i = Var_x[mu_i(x)] / E_x[sigma_i^2(x)]

posterior_variance = np.exp(latent_logvar.astype(np.float64))
mean_posterior_variance = posterior_variance.mean(axis=0)

latent_snr = activity_scores / np.maximum(
    mean_posterior_variance,
    np.finfo(np.float64).tiny,
)

signal_dominated = latent_snr >= 1.0
snr_colors = np.where(signal_dominated, "tab:blue", "tab:gray")

fig, ax = plt.subplots(figsize=(13, 4.8), constrained_layout=True)
ax.bar(
    latent_features,
    latent_snr,
    color=snr_colors,
    edgecolor="black",
    linewidth=0.4,
)

ax.axhline(
    1.0,
    color="tab:red",
    linestyle="--",
    linewidth=1.2,
    label=r"$\mathrm{Var}_x[\mu_i] = \mathbb{E}_x[\sigma_i^2]$",
)

#ax.set_yscale("log")
ax.set_xticks(latent_features)
ax.set_xlabel("latent feature index")
ax.set_ylabel(
    r"$\mathrm{Var}_x[\mu_i(x)] / \mathbb{E}_x[\sigma_i^2(x)]$ (log scale)"
)
ax.set_title(
    f"Latent signal-to-noise ratio ({LATENT_N_EVENTS:,} events); "
    f"{signal_dominated.sum()}/{latent_snr.size} features above 1"
)

ax.grid(axis="y", which="both", alpha=0.25)
ax.legend()

print("SNR >= 1 features:", latent_features[signal_dominated].tolist())
print("SNR < 1 features :", latent_features[~signal_dominated].tolist())

plt.show()

In [ ]:
# Compare mu and sigma distributions for selected active features in u-jet and g-jet samples.
UG_N_EVENTS = min(10_000, latent_mu.shape[0])
UG_BATCH_SIZE = LATENT_BATCH_SIZE
UG_RANDOM_SEED = 1
N_ACTIVE_FEATURES_TO_PLOT = 4
UG_HIST_BINS = 60

G_DATA_PATH = DATA_PATH.with_name(DATA_PATH.name.replace("u_jet_", "g_jet_", 1))
if not G_DATA_PATH.exists():
    G_DATA_PATH = PROJECT_ROOT / "data" / G_DATA_PATH.name
assert G_DATA_PATH.exists(), f"g-jet data file not found: {G_DATA_PATH}"

g_events = load_events_from_dat(str(G_DATA_PATH))
ug_n_events = min(UG_N_EVENTS, len(g_events))
g_rng = np.random.default_rng(UG_RANDOM_SEED)
g_event_indices = g_rng.choice(len(g_events), size=ug_n_events, replace=False)
g_mu_batches, g_logvar_batches = [], []

for start in range(0, ug_n_events, UG_BATCH_SIZE):
    batch_indices = g_event_indices[start:start + UG_BATCH_SIZE]
    batch_images = np.stack([
        model_input_from_event_image(
            make_event_image(g_events[i], pid_lookup, pid_min, n_channels, img_size, checkpoint),
            checkpoint,
        )
        for i in batch_indices
    ])
    x_batch = torch.from_numpy(batch_images).to(DEVICE)
    with torch.no_grad():
        g_mu_batch, g_logvar_batch = model.encode(x_batch)
    g_mu_batches.append(g_mu_batch.cpu().numpy())
    g_logvar_batches.append(g_logvar_batch.cpu().numpy())

u_mu = latent_mu[:ug_n_events]
u_sigma = np.exp(0.5 * latent_logvar[:ug_n_events].astype(np.float64))
g_mu = np.concatenate(g_mu_batches, axis=0)
g_logvar = np.concatenate(g_logvar_batches, axis=0)
g_sigma = np.exp(0.5 * g_logvar.astype(np.float64))

active_feature_indices = latent_features[is_active]
assert active_feature_indices.size > 0, "No active feature passed ACTIVE_THRESHOLD."
selected_features = active_feature_indices[
    np.argsort(activity_scores[active_feature_indices])[::-1][:N_ACTIVE_FEATURES_TO_PLOT]
]

def common_hist_edges(values_a, values_b, n_bins):
    combined = np.concatenate([values_a, values_b])
    low, high = np.quantile(combined, [0.005, 0.995])
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low, high = float(combined.min()), float(combined.max())
    if high <= low:
        high = low + 1e-12
    return np.linspace(low, high, n_bins + 1)

fig, axes = plt.subplots(
    len(selected_features), 2,
    figsize=(12, 2.8 * len(selected_features)),
    squeeze=False, constrained_layout=True,
)
for row, feature_idx in enumerate(selected_features):
    mu_edges = common_hist_edges(u_mu[:, feature_idx], g_mu[:, feature_idx], UG_HIST_BINS)
    sigma_edges = common_hist_edges(u_sigma[:, feature_idx], g_sigma[:, feature_idx], UG_HIST_BINS)

    axes[row, 0].hist(u_mu[:, feature_idx], bins=mu_edges, density=True, histtype="step", linewidth=1.5, label="u jet")
    axes[row, 0].hist(g_mu[:, feature_idx], bins=mu_edges, density=True, histtype="step", linewidth=1.5, label="g jet")
    axes[row, 0].set_title(rf"feature {feature_idx}: $\mu_i(x)$")
    axes[row, 0].set_xlabel(r"$\mu_i$")

    axes[row, 1].hist(u_sigma[:, feature_idx], bins=sigma_edges, density=True, histtype="step", linewidth=1.5, label="u jet")
    axes[row, 1].hist(g_sigma[:, feature_idx], bins=sigma_edges, density=True, histtype="step", linewidth=1.5, label="g jet")
    axes[row, 1].set_title(rf"feature {feature_idx}: $\sigma_i(x)$")
    axes[row, 1].set_xlabel(r"$\sigma_i$")

    for ax in axes[row]:
        ax.set_ylabel("density")
        ax.grid(alpha=0.2)
        ax.legend()

fig.suptitle(
    f"u/g distributions for the {len(selected_features)} most active latent features "
    f"({ug_n_events:,} events per class)",
    fontsize=14,
)
print("selected active features:", selected_features.tolist())
plt.show()

In [ ]:
# 1D histograms of mu and sigma for selected latent features.
# Requires previous cell variables:
#   u_mu, u_sigma, g_mu, g_sigma

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# You can freely edit this part.
#
# Feature indices are zero-based:
#   first latent feature  -> 0
#   second latent feature -> 1
#   ...
# ------------------------------------------------------------

LATENT_1D_FEATURES = [11, 2, 3]   # e.g. [0], [0, 1, 2], [12, 14]
LATENT_1D_QUANTITIES = ["mu", "sigma"]  # choose from ["mu"], ["sigma"], or ["mu", "sigma"]

LATENT_1D_BINS = 70
LATENT_1D_QUANTILE_RANGE = (0.005, 0.995)
LATENT_1D_DENSITY = True
LATENT_1D_HISTTYPE = "step"     # "step" or "bar"
LATENT_1D_LINEWIDTH = 1.6

required_vars = ["u_mu", "u_sigma", "g_mu", "g_sigma"]
missing_vars = [name for name in required_vars if name not in globals()]
assert not missing_vars, f"Missing variables from previous cell: {missing_vars}"

latent_dim = u_mu.shape[1]
for feature_idx in LATENT_1D_FEATURES:
    assert 0 <= feature_idx < latent_dim, (
        f"feature index {feature_idx} out of range; latent dim = {latent_dim}"
    )

for quantity in LATENT_1D_QUANTITIES:
    assert quantity in {"mu", "sigma"}, 'LATENT_1D_QUANTITIES must contain only "mu" or "sigma".'

def get_1d_latent_quantity(mu_array, sigma_array, feature_idx, quantity):
    if quantity == "mu":
        return mu_array[:, feature_idx]
    if quantity == "sigma":
        return sigma_array[:, feature_idx]
    raise ValueError('quantity must be "mu" or "sigma"')

def common_hist_edges(values_a, values_b, n_bins=LATENT_1D_BINS, qrange=LATENT_1D_QUANTILE_RANGE):
    combined = np.concatenate([values_a, values_b])
    low, high = np.quantile(combined, qrange)

    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low, high = float(np.min(combined)), float(np.max(combined))

    if high <= low:
        high = low + 1e-12

    return np.linspace(low, high, n_bins + 1)

n_rows = len(LATENT_1D_FEATURES)
n_cols = len(LATENT_1D_QUANTITIES)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(5.5 * n_cols, 3.6 * n_rows),
    squeeze=False,
    constrained_layout=True,
)

for row, feature_idx in enumerate(LATENT_1D_FEATURES):
    for col, quantity in enumerate(LATENT_1D_QUANTITIES):
        ax = axes[row, col]

        u_values = get_1d_latent_quantity(u_mu, u_sigma, feature_idx, quantity)
        g_values = get_1d_latent_quantity(g_mu, g_sigma, feature_idx, quantity)

        bins = common_hist_edges(u_values, g_values)

        ax.hist(
            u_values,
            bins=bins,
            density=LATENT_1D_DENSITY,
            histtype=LATENT_1D_HISTTYPE,
            linewidth=LATENT_1D_LINEWIDTH,
            alpha=0.9,
            label="u jet",
        )
        ax.hist(
            g_values,
            bins=bins,
            density=LATENT_1D_DENSITY,
            histtype=LATENT_1D_HISTTYPE,
            linewidth=LATENT_1D_LINEWIDTH,
            alpha=0.9,
            label="g jet",
        )

        u_mean = np.mean(u_values)
        u_std = np.std(u_values, ddof=1)
        g_mean = np.mean(g_values)
        g_std = np.std(g_values, ddof=1)

        ax.axvline(u_mean, linestyle="--", linewidth=1.2, alpha=0.8)
        ax.axvline(g_mean, color="orange", linestyle="--", linewidth=1.2, alpha=0.8)

        ax.set_title(
            rf"feature {feature_idx}: ${quantity}_{{{feature_idx}}}$"
            "\n"
            rf"u: mean={u_mean:.4g}, std={u_std:.4g}; "
            rf"g: mean={g_mean:.4g}, std={g_std:.4g}"
        )
        ax.set_xlabel(rf"${quantity}_{{{feature_idx}}}$")
        ax.set_ylabel("density" if LATENT_1D_DENSITY else "count")
        ax.grid(alpha=0.2)
        ax.legend()

fig.suptitle("1D latent-feature distributions", fontsize=14)
plt.show()

In [ ]:
# 2D distributions of selected latent features.
# Requires previous cell variables:
#   u_mu, u_sigma, g_mu, g_sigma, selected_features

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# You can freely edit this part.
#
# Format:
#   (x_feature, x_quantity, y_feature, y_quantity)
#
# quantity must be "mu", "sigma", or "logvar".
#
# Examples:
#   (0, "mu", 1, "mu")       -> x = mu_0,     y = mu_1
#   (0, "mu", 1, "sigma")    -> x = mu_0,     y = sigma_1
#   (3, "sigma", 7, "mu")    -> x = sigma_3,  y = mu_7
# ------------------------------------------------------------

if "selected_features" in globals() and len(selected_features) >= 2:
    f0, f1 = int(selected_features[0]), int(selected_features[1])
else:
    f0, f1 = 0, 1

LATENT_2D_PAIRS = [
    (f0, "mu",    f1, "mu"),
    (f0, "mu",    f1, "sigma"),
    (f0, "sigma", f1, "sigma"),
]

LATENT_2D_PAIRS = [
    (11, "mu",    15, "mu"),
    (11, "mu",    27, "mu"),
    (11, "mu",    30, "mu"),
    (11, "mu",    1, "mu"),

]

LATENT_2D_BINS = 70
LATENT_2D_QUANTILE_RANGE = (0.005, 0.995)
LATENT_2D_PLOT_MODE = "hist2d"  # "hist2d" or "scatter"
LATENT_2D_SCATTER_ALPHA = 0.12
LATENT_2D_MAX_POINTS = 5000     # only used for scatter mode

required_vars = ["u_mu", "u_sigma", "g_mu", "g_sigma"]
missing_vars = [name for name in required_vars if name not in globals()]
assert not missing_vars, f"Missing variables from previous cell: {missing_vars}"

def get_latent_quantity(mu_array, sigma_array, feature_idx, quantity):
    if quantity == "mu":
        return mu_array[:, feature_idx]
    if quantity == "sigma":
        return sigma_array[:, feature_idx]
    if quantity == "logvar":
        sigma = sigma_array[:, feature_idx]
        return 2.0 * np.log(sigma)
    raise ValueError('quantity must be "mu", "sigma", or "logvar"')

def robust_limits(values_a, values_b, qrange=LATENT_2D_QUANTILE_RANGE):
    combined = np.concatenate([values_a, values_b])
    low, high = np.quantile(combined, qrange)

    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low, high = float(np.min(combined)), float(np.max(combined))

    if high <= low:
        high = low + 1e-12

    margin = 0.05 * (high - low)
    return low - margin, high + margin

n_pairs = len(LATENT_2D_PAIRS)

fig, axes = plt.subplots(
    n_pairs, 2,
    figsize=(11, 4.2 * n_pairs),
    squeeze=False,
    constrained_layout=True,
)

for row, (x_feature, x_quantity, y_feature, y_quantity) in enumerate(LATENT_2D_PAIRS):
    u_x = get_latent_quantity(u_mu, u_sigma, x_feature, x_quantity)
    u_y = get_latent_quantity(u_mu, u_sigma, y_feature, y_quantity)

    g_x = get_latent_quantity(g_mu, g_sigma, x_feature, x_quantity)
    g_y = get_latent_quantity(g_mu, g_sigma, y_feature, y_quantity)

    u_cov = np.cov(u_x, u_y, ddof=1)[0, 1]
    g_cov = np.cov(g_x, g_y, ddof=1)[0, 1]

    u_corr = np.corrcoef(u_x, u_y)[0, 1]
    g_corr = np.corrcoef(g_x, g_y)[0, 1]

    print(
        f"{x_quantity}_{x_feature} vs {y_quantity}_{y_feature}\n"
        f"  u jet: cov = {u_cov:.6g}, corr = {u_corr:.6g}\n"
        f"  g jet: cov = {g_cov:.6g}, corr = {g_corr:.6g}"
    )

    xlim = robust_limits(u_x, g_x)
    ylim = robust_limits(u_y, g_y)

    xlabel = rf"${x_quantity}_{{{x_feature}}}$"
    ylabel = rf"${y_quantity}_{{{y_feature}}}$"

    for ax, class_name, x_values, y_values in [
        (axes[row, 0], "u jet", u_x, u_y),
        (axes[row, 1], "g jet", g_x, g_y),
    ]:
        if LATENT_2D_PLOT_MODE == "hist2d":
            h = ax.hist2d(
                x_values,
                y_values,
                bins=LATENT_2D_BINS,
                range=[xlim, ylim],
                density=True,
                cmap="viridis",
            )
            fig.colorbar(h[3], ax=ax, label="density")

        elif LATENT_2D_PLOT_MODE == "scatter":
            if len(x_values) > LATENT_2D_MAX_POINTS:
                rng = np.random.default_rng(0)
                idx = rng.choice(len(x_values), size=LATENT_2D_MAX_POINTS, replace=False)
                x_plot = x_values[idx]
                y_plot = y_values[idx]
            else:
                x_plot = x_values
                y_plot = y_values

            ax.scatter(
                x_plot,
                y_plot,
                s=6,
                alpha=LATENT_2D_SCATTER_ALPHA,
            )

        else:
            raise ValueError('LATENT_2D_PLOT_MODE must be "hist2d" or "scatter"')

        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{class_name}: {xlabel} vs {ylabel}")
        ax.grid(alpha=0.2)

fig.suptitle("2D latent-space distributions for selected active features", fontsize=14)
plt.show()

In [ ]:
# 2D distributions of selected latent features.
# Requires previous cell variables:
#   u_mu, u_sigma, g_mu, g_sigma, selected_features

import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# You can freely edit this part.
#
# Format:
#   (x_feature, x_quantity, y_feature, y_quantity)
#
# quantity must be "mu" or "sigma".
#
# Examples:
#   (0, "mu", 1, "mu")       -> x = mu_0,     y = mu_1
#   (0, "mu", 1, "sigma")    -> x = mu_0,     y = sigma_1
#   (3, "sigma", 7, "mu")    -> x = sigma_3,  y = mu_7
# ------------------------------------------------------------

if "selected_features" in globals() and len(selected_features) >= 2:
    f0, f1 = int(selected_features[0]), int(selected_features[1])
else:
    f0, f1 = 0, 1

LATENT_2D_PAIRS = [
    (11, "mu",    1, "logvar"),
    (11, "logvar",    27, "logvar"),
    (11, "logvar",    30, "logvar"),
    (11, "sigma",    1, "sigma"),

]

LATENT_2D_BINS = 70
LATENT_2D_QUANTILE_RANGE = (0.005, 0.995)
LATENT_2D_PLOT_MODE = "hist2d"  # "hist2d" or "scatter"
LATENT_2D_SCATTER_ALPHA = 0.12
LATENT_2D_MAX_POINTS = 5000     # only used for scatter mode

required_vars = ["u_mu", "u_sigma", "g_mu", "g_sigma"]
missing_vars = [name for name in required_vars if name not in globals()]
assert not missing_vars, f"Missing variables from previous cell: {missing_vars}"

def get_latent_quantity(mu_array, sigma_array, feature_idx, quantity):
    if quantity == "mu":
        return mu_array[:, feature_idx]
    if quantity == "sigma":
        return sigma_array[:, feature_idx]
    if quantity == "logvar":
        sigma = sigma_array[:, feature_idx]
        return 2.0 * np.log(sigma)
    raise ValueError('quantity must be "mu", "sigma", or "logvar"')

def robust_limits(values_a, values_b, qrange=LATENT_2D_QUANTILE_RANGE):
    combined = np.concatenate([values_a, values_b])
    low, high = np.quantile(combined, qrange)

    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        low, high = float(np.min(combined)), float(np.max(combined))

    if high <= low:
        high = low + 1e-12

    margin = 0.05 * (high - low)
    return low - margin, high + margin

n_pairs = len(LATENT_2D_PAIRS)

fig, axes = plt.subplots(
    n_pairs, 2,
    figsize=(11, 4.2 * n_pairs),
    squeeze=False,
    constrained_layout=True,
)

for row, (x_feature, x_quantity, y_feature, y_quantity) in enumerate(LATENT_2D_PAIRS):
    u_x = get_latent_quantity(u_mu, u_sigma, x_feature, x_quantity)
    u_y = get_latent_quantity(u_mu, u_sigma, y_feature, y_quantity)

    g_x = get_latent_quantity(g_mu, g_sigma, x_feature, x_quantity)
    g_y = get_latent_quantity(g_mu, g_sigma, y_feature, y_quantity)

    u_cov = np.cov(u_x, u_y, ddof=1)[0, 1]
    g_cov = np.cov(g_x, g_y, ddof=1)[0, 1]

    u_corr = np.corrcoef(u_x, u_y)[0, 1]
    g_corr = np.corrcoef(g_x, g_y)[0, 1]

    print(
        f"{x_quantity}_{x_feature} vs {y_quantity}_{y_feature}\n"
        f"  u jet: cov = {u_cov:.6g}, corr = {u_corr:.6g}\n"
        f"  g jet: cov = {g_cov:.6g}, corr = {g_corr:.6g}"
    )

    xlim = robust_limits(u_x, g_x)
    ylim = robust_limits(u_y, g_y)

    xlabel = rf"${x_quantity}_{{{x_feature}}}$"
    ylabel = rf"${y_quantity}_{{{y_feature}}}$"

    for ax, class_name, x_values, y_values in [
        (axes[row, 0], "u jet", u_x, u_y),
        (axes[row, 1], "g jet", g_x, g_y),
    ]:
        if LATENT_2D_PLOT_MODE == "hist2d":
            h = ax.hist2d(
                x_values,
                y_values,
                bins=LATENT_2D_BINS,
                range=[xlim, ylim],
                density=True,
                cmap="viridis",
            )
            fig.colorbar(h[3], ax=ax, label="density")

        elif LATENT_2D_PLOT_MODE == "scatter":
            if len(x_values) > LATENT_2D_MAX_POINTS:
                rng = np.random.default_rng(0)
                idx = rng.choice(len(x_values), size=LATENT_2D_MAX_POINTS, replace=False)
                x_plot = x_values[idx]
                y_plot = y_values[idx]
            else:
                x_plot = x_values
                y_plot = y_values

            ax.scatter(
                x_plot,
                y_plot,
                s=6,
                alpha=LATENT_2D_SCATTER_ALPHA,
            )

        else:
            raise ValueError('LATENT_2D_PLOT_MODE must be "hist2d" or "scatter"')

        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{class_name}: {xlabel} vs {ylabel}")
        ax.grid(alpha=0.2)

fig.suptitle("2D latent-space distributions for selected active features", fontsize=14)
plt.show()

In [ ]:
# Inspect all latent posterior means for one selected u-jet or g-jet event.
LATENT_SAMPLE_TYPE = "u"  # "u" or "g"
LATENT_SAMPLE_INDEX = 0

sample_type = LATENT_SAMPLE_TYPE.lower()
assert sample_type in {"u", "g"}, 'LATENT_SAMPLE_TYPE must be "u" or "g".'
if sample_type == "u":
    sample_events = events
else:
    if "g_events" not in globals():
        sample_g_path = DATA_PATH.with_name(DATA_PATH.name.replace("u_jet_", "g_jet_", 1))
        if not sample_g_path.exists():
            sample_g_path = PROJECT_ROOT / "data" / sample_g_path.name
        assert sample_g_path.exists(), f"g-jet data file not found: {sample_g_path}"
        g_events = load_events_from_dat(str(sample_g_path))
    sample_events = g_events

assert 0 <= LATENT_SAMPLE_INDEX < len(sample_events), (
    f"LATENT_SAMPLE_INDEX must be in [0, {len(sample_events) - 1}]"
)
sample_image = make_event_image(
    sample_events[LATENT_SAMPLE_INDEX], pid_lookup, pid_min,
    n_channels, img_size, checkpoint,
)
sample_model_input = model_input_from_event_image(sample_image, checkpoint)
sample_x = torch.from_numpy(sample_model_input[None]).to(DEVICE)
with torch.no_grad():
    sample_mu_tensor, _ = model.encode(sample_x)
sample_mu = sample_mu_tensor.cpu().numpy()[0]

print(f"sample: {sample_type}-jet event {LATENT_SAMPLE_INDEX}")
print(f"latent dimension: {sample_mu.size}")
print("mu =")
print(np.array2string(
    sample_mu, precision=5, separator=", ", suppress_small=False, max_line_width=160
))